# Stage 2 Notebook 69 - Exp2NNN Anchor + cls_sep + 5-epoch head_warmup + full 70K + 14ep

**Sequential training: heads first, then backbone.** NB63 (DN-DETR + ultra-stable) showed that during head_warmup (backbone frozen), val_lane_best_f1 hit 0.630 at epoch 1 -- the highest cls discrimination ever measured -- then DEGRADED when backbone unfroze at epoch 4. The head's converged cls patterns were destroyed by backbone updates.

Exp2NNN applies this insight to the anchor head: extend head_warmup from 1 epoch (NB62) to 5 epochs. The head fully converges on the frozen-random backbone (still does mask aux training, still does geometry training within head's params) BEFORE the backbone is allowed to update. Then 9 epochs of full_finetune with the now-converged head.

Single diff vs NB62 (exp57):
- `phases.head_warmup until_epoch: 1 -> 5`
- `end_epoch: 12 -> 14` (compensate for the longer warmup phase)
- `lr_scheduler.warmup_epochs: 1 -> 2`

### Run mode
1. Smoke.
2. 14 epochs full 70K. ~3-3.5 hr.

In [4]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [5]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp64_rmt_gca_anchor_cls_sep_vfl_long_warmup_full_data_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp64_rmt_gca_anchor_cls_sep_vfl_long_warmup_full_data_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp64_rmt_gca_anchor_cls_sep_vfl_long_warmup_full_data_joint_smoke.log
OK exp64_rmt_gca_anchor_cls_sep_vfl_long_warmup_full_data_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=4.6579 det_loss=3.5420 grad_cos=0.3695 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.49797913432121277, 'gate/lane_mean': 0.4954921007156372, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [6]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp64_rmt_gca_anchor_cls_sep_vfl_long_warmup_full_data_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'full14'
    EPOCHS = 14
    BATCH_SIZE = 8
    LIMIT_TRAIN = None
    LIMIT_VAL = 2000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: None
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp64_rmt_gca_anchor_cls_sep_vfl_long_warmup_full_data_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp64_rmt_gca_anchor_cls_sep_vfl_long_warmup_full_data_joint_full14 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp64_rmt_gca_anchor_cls_sep_vfl_long_warmup_full_data_joint_full14.tar --epochs 14 --batch-size 8 --limit-val 2000 --force-extract --print-every 50
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp64_rmt_gca_anchor_cls_sep_vfl_long_warmup_full_data_joint_full14.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp64_rmt_gca_anchor_cls_sep_vfl_long_warmup_full_data_joint_full14_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config

0

## What to watch in Exp2NNN

Reference NB62 (1-ep head_warmup, 12 ep total): matched_iou=0.550, decoded_f1=0.073, gap=0.045.
Reference NB63 (DN-DETR with peak in head_warmup): val_lane_best_f1=0.630 at ep1 (decayed).

Pass criteria at epoch 14:
- **In epochs 1-5 (head_warmup)**: val_lane_f1 climbing steadily, val_lane_best_f1 should approach 0.20 by epoch 5 (vs NB62 ep1=0.000).
- **In epochs 6-14 (full_finetune)**: cls preserved, geometry climbing as backbone unblocks.
- val/matched_line_iou >= 0.55 (preserve NB62 geometry).
- val/lane/decoded_f1 >= 0.10 (40 % over NB62; preserved cls + improved geometry).
- pos-neg gap >= 0.06.